# Semana 1 — Listas, diccionarios y conjuntos aplicado a Penonomé

Aplica las unidades 10 a 16 del curso (listas y tuplas, diccionarios y JSON, iteración y búsqueda, listas bidimensionales, CRUD de diccionarios, diccionarios avanzados y conjuntos) a los datos oficiales del Censo 2023, al índice de pobreza multidimensional del PNUD y a las afectaciones del sismo del 9 de octubre de 2026 que publica SINAPROC, sin pandas.

[Proyecto](../../README.md) | [Anterior](../../README.md) | [Siguiente](s02-funciones-clausuras-clases.ipynb)

In [1]:
def verificar(descripcion, obtenido, esperado):
    """Compara el resultado de un ejercicio con el esperado e informa sin detener el notebook."""
    if obtenido == esperado:
        print(f"Correcto: {descripcion}")
    else:
        print(f"Incorrecto: {descripcion}. Se esperaba {esperado!r} y se obtuvo {obtenido!r}")

## 0. Leer los CSV con el módulo `csv`

Los datos viven en `datos/penonome/`. Se leen con `open` dentro de un bloque `with` (unidad 13, [lectura y escritura de archivos](../../../../01-python/semana-01/clase-1-listas-tuplas-diccionarios/u13-listas-bidimensionales.ipynb)) y con `csv.DictReader`, que entrega cada fila como un diccionario cuyas claves son los encabezados. El resultado es una **lista de diccionarios**.

**Clave:** los CSV del paquete tienen BOM (marca de orden de bytes); por eso se abren con `encoding="utf-8-sig"`. Con `utf-8` la primera clave quedaría como `"\ufeffid_corregimiento"`.

El cargador completo está en `src/penonome/carga.py`; la celda siguiente empieza por su idea central.

Los cargadores de `penonome.carga` hacen lo mismo y, además, convierten las columnas numéricas con `int` o `float`. Una celda vacía (población de 2010 de un corregimiento que aún no existía) queda como `None`.

In [2]:
import csv

from penonome import PAQUETE

with open(PAQUETE / "00_catalogo_corregimientos.csv", encoding="utf-8-sig", newline="") as archivo:
    catalogo = list(csv.DictReader(archivo))

print(type(catalogo), type(catalogo[0]))
print(catalogo[0])
verificar("el catálogo tiene 16 corregimientos", len(catalogo), 16)

from penonome.carga import (PADRES_LEY_296, cargar_afectaciones, cargar_ipm, cargar_poblacion,
                            cargar_registros_danos, cargar_viviendas)

poblacion = cargar_poblacion()                      # tabla 01
viviendas = cargar_viviendas("corregimiento")       # tabla 02, solo corregimientos
distrito = cargar_viviendas("distrito")[0]          # fila total de Penonomé
provincia = cargar_viviendas("provincia")[0]        # fila total de Coclé

print(poblacion[3])
verificar("Coclé tiene 77,384 viviendas en el Censo 2023", provincia["viviendas"], 77384)

<class 'list'> <class 'dict'>
{'id_corregimiento': 'penonome_cabecera', 'provincia': 'Coclé', 'distrito': 'Penonomé', 'corregimiento': 'Penonomé (cabecera)', 'nota_cambio_limites': ''}
Correcto: el catálogo tiene 16 corregimientos
{'id_corregimiento': 'chiguiri_arriba', 'provincia': 'Coclé', 'distrito': 'Penonomé', 'corregimiento': 'Chiguirí Arriba', 'superficie_km2': 104, 'pob_2000': 8581, 'pob_2010': 10018, 'pob_2023': 7185, 'densidad_2023': 69.1, 'nota': 'dividido por la Ley 296 de 26-abr-2022 (corregimientos nuevos)'}
Correcto: Coclé tiene 77,384 viviendas en el Censo 2023


## 1. Listas y tuplas: población 2023

Una lista guarda una secuencia ordenada y modificable; una tupla, una secuencia inmutable ([unidad 10](../../../../01-python/semana-01/clase-1-listas-tuplas-diccionarios/u10-listas-y-tuplas.ipynb)). Aquí cada corregimiento se resume en una tupla `(id, nombre, población 2023)`, porque esos tres datos no deben cambiar, y las tuplas se agrupan en una lista.

`sorted` con `reverse=True` ordena de mayor a menor y el rebanado `[:3]` toma los tres primeros. El operador `in` comprueba la pertenencia.

In [3]:
registros = [(fila["id_corregimiento"], fila["corregimiento"], fila["pob_2023"]) for fila in poblacion]

total_2023 = sum(habitantes for _, _, habitantes in registros)
print("Corregimientos:", len(registros))
print("Primero:", registros[0])
print("Último:", registros[-1])
print(f"Población del distrito en 2023: {total_2023:,}")

verificar("los 16 corregimientos suman 104,326 habitantes", total_2023, 104326)
verificar("la suma coincide con la fila distrito de la tabla 02", total_2023, distrito["poblacion"])

por_poblacion = sorted(registros, key=lambda registro: registro[2], reverse=True)
tres_mayores = [nombre for _, nombre, _ in por_poblacion[:3]]
print("Los tres corregimientos más poblados:", tres_mayores)

nombres = [nombre for _, nombre, _ in registros]
print("¿Está 'Riecito' en la lista?", "Riecito" in nombres)

cabecera = registros[0]
try:
    cabecera[2] = 0
except TypeError as error:
    print("Una tupla no se modifica:", error)

verificar("tres más poblados", tres_mayores, ["Penonomé (cabecera)", "El Coco", "Cañaveral"])

Corregimientos: 16
Primero: ('penonome_cabecera', 'Penonomé (cabecera)', 25836)
Último: ('san_miguel', 'San Miguel', 2038)
Población del distrito en 2023: 104,326
Correcto: los 16 corregimientos suman 104,326 habitantes
Correcto: la suma coincide con la fila distrito de la tabla 02
Los tres corregimientos más poblados: ['Penonomé (cabecera)', 'El Coco', 'Cañaveral']
¿Está 'Riecito' en la lista? True
Una tupla no se modifica: 'tuple' object does not support item assignment
Correcto: tres más poblados


## 2. Iteración y búsqueda: ¿qué corregimiento creció más entre 2010 y 2023?

La [unidad 12](../../../../01-python/semana-01/clase-1-listas-tuplas-diccionarios/u12-iteracion-y-busqueda.ipynb) recorre secuencias con `for` y busca valores. Antes de comparar hay que filtrar: según la Ley 296 de 2022, cinco corregimientos se dividieron y seis son nuevos, así que solo los que no cambiaron de límites tienen una población de 2010 comparable con la de 2023.

**Error común:** calcular el crecimiento de Pajonal sin más da -43.4 %, pero esa caída no es despoblación: en 2022 se le separó General Victoriano Lorenzo.

In [4]:
divididos = list(PADRES_LEY_296)                         # los cinco que perdieron territorio
nuevos = [hijo for hijos in PADRES_LEY_296.values() for hijo in hijos]

crecimiento = []                                         # lista de tuplas (nombre, %)
for fila in poblacion:
    clave = fila["id_corregimiento"]
    if clave in divididos or clave in nuevos:
        continue
    porcentaje = round((fila["pob_2023"] / fila["pob_2010"] - 1) * 100, 1)
    crecimiento.append((fila["corregimiento"], porcentaje))

for nombre, porcentaje in crecimiento:
    print(f"{nombre:<22}{porcentaje:>7.1f} %")

Penonomé (cabecera)      18.8 %
Cañaveral                24.0 %
Coclé                    41.7 %
El Coco                  97.4 %
Río Grande               12.4 %


Búsqueda lineal del máximo, sin `max`: se guarda el mejor candidato y se compara con cada elemento. Luego se confirma con `max` y una función clave.

Una búsqueda lineal también sirve para encontrar una fila por su nombre. Si no la encuentra, devuelve `None` en lugar de fallar.

In [5]:
mayor = crecimiento[0]
for candidato in crecimiento[1:]:
    if candidato[1] > mayor[1]:
        mayor = candidato

print("Mayor crecimiento (búsqueda lineal):", mayor)
print("Mayor crecimiento (max):", max(crecimiento, key=lambda par: par[1]))

verificar("El Coco casi duplicó su población (97.4 %)", mayor, ("El Coco", 97.4))

def buscar_por_nombre(filas, nombre):
    """Recorre las filas y devuelve la primera cuyo corregimiento coincide, o None."""
    for fila in filas:
        if fila["corregimiento"] == nombre:
            return fila
    return None

riecito = buscar_por_nombre(poblacion, "Riecito")
print(riecito["corregimiento"], riecito["pob_2010"], riecito["pob_2023"])
print(buscar_por_nombre(poblacion, "Antón"))

verificar("Riecito no tiene población de 2010 (no existía)", riecito["pob_2010"], None)

Mayor crecimiento (búsqueda lineal): ('El Coco', 97.4)
Mayor crecimiento (max): ('El Coco', 97.4)
Correcto: El Coco casi duplicó su población (97.4 %)
Riecito None 1315
None
Correcto: Riecito no tiene población de 2010 (no existía)


## 3. Listas bidimensionales: tabla de acceso a internet

Una lista bidimensional es una lista de filas; `tabla[i][j]` lee la fila `i`, columna `j` ([unidad 13](../../../../01-python/semana-01/clase-1-listas-tuplas-diccionarios/u13-listas-bidimensionales.ipynb)). Se arma con una comprensión de listas a partir de la tabla 02: viviendas, viviendas sin internet y su porcentaje.

Con un doble bucle se suman las columnas numéricas (1 y 2). La suma debe coincidir con la fila del distrito: es la misma suma de control que usa el paquete.

In [6]:
tabla = [
    [fila["area"], fila["viviendas"], fila["sin_internet"], round(fila["sin_internet"] / fila["viviendas"] * 100, 1)]
    for fila in viviendas
]

print(f"{'Corregimiento':<28}{'Viviendas':>10}{'Sin internet':>14}{'%':>7}")
for nombre, total, sin_internet, porcentaje in tabla:
    print(f"{nombre:<28}{total:>10,}{sin_internet:>14,}{porcentaje:>7.1f}")

totales = [0, 0]
for fila in tabla:
    for j in range(1, 3):
        totales[j - 1] += fila[j]

porcentaje_distrito = round(totales[1] / totales[0] * 100, 1)
print("Viviendas:", totales[0], "| Sin internet:", totales[1], "|", porcentaje_distrito, "%")

verificar("las viviendas de los corregimientos suman 28,547", totales[0], distrito["viviendas"])
verificar("el 41.9 % de las viviendas del distrito no tiene internet", porcentaje_distrito, 41.9)
verificar("celda tabla[7][3]: Río Indio, 99.0 % sin internet", tabla[7][3], 99.0)

Corregimiento                Viviendas  Sin internet      %
Penonomé (cabecera)              7,547         1,323   17.5
Cañaveral                        2,624           828   31.6
Coclé                            1,722           451   26.2
Chiguirí Arriba                  1,712         1,319   77.0
El Coco                          3,271           636   19.4
Pajonal                          1,935         1,115   57.6
Río Grande                       1,090           383   35.1
Río Indio                          953           943   99.0
Toabré                           2,051         1,317   64.2
Tulú                               508           505   99.4
Boca de Tucué                      518           511   98.6
Candelario Ovalle                  758           593   78.2
General Victoriano Lorenzo       2,369           761   32.1
Las Minas                          733           545   74.4
Riecito                            281           276   98.2
San Miguel                         475  

## 4. Diccionarios y JSON

Un diccionario guarda pares clave-valor y permite leer un dato por su clave sin recorrer la lista ([unidad 11](../../../../01-python/semana-01/clase-1-listas-tuplas-diccionarios/u11-diccionarios-y-json.ipynb)). La clave natural del paquete es `id_corregimiento`, que une las tablas 01 y 02.

In [7]:
corregimientos = {}
for fila in poblacion:
    corregimientos[fila["id_corregimiento"]] = {
        "nombre": fila["corregimiento"],
        "superficie_km2": fila["superficie_km2"],
        "pob_2010": fila["pob_2010"],
        "pob_2023": fila["pob_2023"],
    }

for fila in viviendas:                       # une la tabla 02 por la misma clave
    datos = corregimientos[fila["id_corregimiento"]]
    datos["viviendas"] = fila["viviendas"]
    datos["sin_internet"] = fila["sin_internet"]

print(corregimientos["el_coco"])
print("{0}: {1:,} habitantes en {2:.1f} km²".format(
    corregimientos["el_coco"]["nombre"], corregimientos["el_coco"]["pob_2023"], corregimientos["el_coco"]["superficie_km2"]))

verificar("El Coco tiene 3,271 viviendas", corregimientos["el_coco"]["viviendas"], 3271)

{'nombre': 'El Coco', 'superficie_km2': 148.3, 'pob_2010': 5605, 'pob_2023': 11064, 'viviendas': 3271, 'sin_internet': 636}
El Coco: 11,064 habitantes en 148.3 km²
Correcto: El Coco tiene 3,271 viviendas


`json.dumps` convierte el diccionario en texto JSON y `json.loads` lo devuelve a diccionario. Se escribe en un archivo temporal con `json.dump` para no dejar archivos nuevos en el proyecto.

**Error común:** JSON no tiene `None`; lo escribe como `null`, y al leerlo vuelve a ser `None`.

In [8]:
import json
import tempfile
from pathlib import Path

texto = json.dumps(corregimientos["riecito"], ensure_ascii=False)
print(texto)

with tempfile.TemporaryDirectory() as carpeta:
    ruta = Path(carpeta) / "corregimientos.json"
    with open(ruta, "w", encoding="utf-8") as archivo:
        json.dump(corregimientos, archivo, ensure_ascii=False, indent=2)
    with open(ruta, encoding="utf-8") as archivo:
        leidos = json.load(archivo)

verificar("el JSON leído es igual al diccionario original", leidos, corregimientos)
verificar("None se guarda como null", '"pob_2010": null' in texto, True)

{"nombre": "Riecito", "superficie_km2": 83.3, "pob_2010": null, "pob_2023": 1315, "viviendas": 281, "sin_internet": 276}
Correcto: el JSON leído es igual al diccionario original
Correcto: None se guarda como null


## 5. CRUD sobre el diccionario de corregimientos

La [unidad 14](../../../../01-python/semana-01/clase-2-diccionarios-y-conjuntos/u14-diccionarios-crud.ipynb) trabaja las cuatro operaciones: crear, leer, actualizar y borrar. Se aplican sobre una copia para no alterar el diccionario de la sección anterior.

**Referencia:** la fila `nivel = distrito` de la tabla 02 es el total de Penonomé; no debe sumarse junto con los corregimientos. El ejemplo la agrega y la retira para mostrar por qué.

In [9]:
import copy

registro = copy.deepcopy(corregimientos)

# Crear: se agrega el total del distrito como si fuera un corregimiento más
registro["total_distrito"] = {"nombre": "Penonomé (distrito)", "pob_2023": distrito["poblacion"]}
print("Con el total, la suma da el doble:", sum(d["pob_2023"] for d in registro.values()))

# Leer: get devuelve un valor por defecto si la clave no existe
print("Superficie de El Coco (km²):", registro.get("el_coco", {}).get("superficie_km2"))
print("Antón:", registro.get("anton", "no es un corregimiento de Penonomé"))

# Actualizar: update agrega la densidad redondeada a la cabecera
cabecera = registro["penonome_cabecera"]
cabecera.update({"densidad": round(cabecera["pob_2023"] / cabecera["superficie_km2"], 1)})
print("Cabecera:", cabecera)

# Borrar: pop retira el total y lo devuelve
retirado = registro.pop("total_distrito")
print("Retirado:", retirado["nombre"])

verificar("con el total incluido la suma se duplica", retirado["pob_2023"] * 2, 208652)
verificar("después de pop quedan 16 corregimientos", len(registro), 16)
verificar("el original no cambió (copia profunda)", "densidad" in corregimientos["penonome_cabecera"], False)

Con el total, la suma da el doble: 208652
Superficie de El Coco (km²): 148.3
Antón: no es un corregimiento de Penonomé
Cabecera: {'nombre': 'Penonomé (cabecera)', 'superficie_km2': 53.4, 'pob_2010': 21748, 'pob_2023': 25836, 'viviendas': 7547, 'sin_internet': 1323, 'densidad': 483.8}
Retirado: Penonomé (distrito)
Correcto: con el total incluido la suma se duplica
Correcto: después de pop quedan 16 corregimientos
Correcto: el original no cambió (copia profunda)


`setdefault` sirve para contar: crea la clave con 0 la primera vez y luego la incrementa. Aquí se cuenta cuántos corregimientos tiene cada tipo de nota en el catálogo.

In [10]:
tipos = {}
for fila in catalogo:
    nota = fila["nota_cambio_limites"]
    if nota.startswith("creado"):
        tipo = "nuevo"
    elif nota.startswith("dividido"):
        tipo = "dividido"
    else:
        tipo = "sin cambios"
    tipos.setdefault(tipo, 0)
    tipos[tipo] += 1

print(tipos)
verificar("6 nuevos, 5 divididos y 5 sin cambios", tipos, {"sin cambios": 5, "dividido": 5, "nuevo": 6})

{'sin cambios': 5, 'dividido': 5, 'nuevo': 6}
Correcto: 6 nuevos, 5 divididos y 5 sin cambios


## 6. Diccionarios avanzados: comprensión, `defaultdict` y diccionario doble

La [unidad 15](../../../../01-python/semana-01/clase-2-diccionarios-y-conjuntos/u15-diccionarios-avanzados.ipynb) añade formas compactas de crear y agrupar. Una **comprensión de diccionario** calcula el porcentaje de viviendas sin internet de cada corregimiento en una sola expresión.

In [11]:
sin_internet = {
    fila["id_corregimiento"]: round(fila["sin_internet"] / fila["viviendas"] * 100, 1)
    for fila in viviendas
}
mayoria_sin_internet = {clave: pct for clave, pct in sin_internet.items() if pct > 90}

print("Más del 90 % de viviendas sin internet:", mayoria_sin_internet)
verificar("cinco corregimientos superan el 90 %", len(mayoria_sin_internet), 5)
verificar("Tulú es el más desconectado (99.4 %)", max(sin_internet, key=sin_internet.get), "tulu")

Más del 90 % de viviendas sin internet: {'rio_indio': 99.0, 'tulu': 99.4, 'boca_de_tucue': 98.6, 'riecito': 98.2, 'san_miguel': 94.9}
Correcto: cinco corregimientos superan el 90 %
Correcto: Tulú es el más desconectado (99.4 %)


`defaultdict(list)` agrupa sin preguntar si la clave existe. Se agrupan los corregimientos nuevos bajo su corregimiento original, y con ese grupo se reconstruye el territorio de 2010: así el crecimiento sí es comparable.

In [12]:
from collections import defaultdict

poblacion_por_id = {fila["id_corregimiento"]: fila for fila in poblacion}

territorio_2010 = defaultdict(list)
for padre, hijos in PADRES_LEY_296.items():
    territorio_2010[padre].append(padre)
    territorio_2010[padre].extend(hijos)

for padre, partes in territorio_2010.items():
    antes = poblacion_por_id[padre]["pob_2010"]
    ahora = sum(poblacion_por_id[parte]["pob_2023"] for parte in partes)
    print(f"{poblacion_por_id[padre]['corregimiento']:<16} 2010: {antes:>6,}  2023: {ahora:>6,}  {(ahora / antes - 1) * 100:>5.1f} %")

pajonal_2023 = sum(poblacion_por_id[parte]["pob_2023"] for parte in territorio_2010["pajonal"])
verificar("Pajonal con General Victoriano Lorenzo suma 16,174 en 2023", pajonal_2023, 16174)

Chiguirí Arriba  2010: 10,018  2023: 10,800    7.8 %
Pajonal          2010: 13,565  2023: 16,174   19.2 %
Río Indio        2010:  5,240  2023:  5,341    1.9 %
Toabré           2010: 10,203  2023: 11,568   13.4 %
Tulú             2010:  4,624  2023:  4,903    6.0 %
Correcto: Pajonal con General Victoriano Lorenzo suma 16,174 en 2023


Un **diccionario doble** (un diccionario dentro de otro) ordena las afectaciones del sismo en Coclé por nivel de daño y por tipo de dato: `por_nivel[nivel][dato]`. Las afectaciones de SINAPROC vienen en formato largo (`cargar_afectaciones()`): una fila por dato, con `categoria`, `subcategoria`, `desagregacion` y `valor`.

**Error común:** el nivel de daño «Total» es el de las viviendas con daño total, no la suma de todos los niveles. Los cinco niveles (Total, Moderado, Parcial, Leve y Menor) suman las viviendas evaluadas; se comprueba abajo.

In [13]:
afectaciones = cargar_afectaciones()

por_nivel = {}
for fila in afectaciones:
    if fila["categoria"] == "Viviendas por nivel de daño":
        dato = "viviendas"
    elif fila["categoria"] == "Atención especial por nivel de daño":
        dato = "discapacidad" if fila["desagregacion"] == "Personas con discapacidad" else "embarazadas"
    else:
        continue
    por_nivel.setdefault(fila["subcategoria"], {})[dato] = fila["valor"]

print(f"{'Nivel de daño':<14}{'Viviendas':>10}{'Discapacidad':>14}{'Embarazadas':>13}")
for nivel, datos in por_nivel.items():
    print(f"{nivel:<14}{datos['viviendas']:>10,}{datos['discapacidad']:>14,}{datos['embarazadas']:>13,}")

def valor_de(subcategoria, desagregacion="Total"):
    """Valor de la primera fila con esa subcategoría y desagregación (búsqueda lineal), o None."""
    for fila in afectaciones:
        if fila["subcategoria"] == subcategoria and fila["desagregacion"] == desagregacion:
            return fila["valor"]
    return None

evaluadas = valor_de("Viviendas evaluadas")
parcial = por_nivel["Parcial"]["viviendas"]
print(f"Viviendas con daño parcial: {parcial:,} de {evaluadas:,} evaluadas ({parcial / evaluadas * 100:.1f} %)")

verificar("los cinco niveles suman las viviendas evaluadas", sum(d["viviendas"] for d in por_nivel.values()), evaluadas)
verificar("la discapacidad por nivel suma el total de personas con discapacidad",
          sum(d["discapacidad"] for d in por_nivel.values()), valor_de("Personas con discapacidad"))
verificar("las embarazadas por nivel suman el total de embarazadas",
          sum(d["embarazadas"] for d in por_nivel.values()), valor_de("Mujeres embarazadas"))
verificar("daño parcial: viviendas, discapacidad y embarazadas", por_nivel["Parcial"],
          {"viviendas": 1750, "discapacidad": 314, "embarazadas": 11})

Nivel de daño  Viviendas  Discapacidad  Embarazadas
Total                414            77            7
Moderado             147            14            2
Parcial            1,750           314           11
Leve                 145            13            1
Menor                 75            13            0
Viviendas con daño parcial: 1,750 de 2,531 evaluadas (69.1 %)
Correcto: los cinco niveles suman las viviendas evaluadas
Correcto: la discapacidad por nivel suma el total de personas con discapacidad
Correcto: las embarazadas por nivel suman el total de embarazadas
Correcto: daño parcial: viviendas, discapacidad y embarazadas


## 7. Conjuntos: corregimientos nuevos, indicadores por año y pobreza frente a daños

Un conjunto guarda elementos sin repetir y sin orden, y permite unión `|`, intersección `&`, diferencia `-` y diferencia simétrica `^` ([unidad 16](../../../../01-python/semana-01/clase-2-diccionarios-y-conjuntos/u16-conjuntos.ipynb)). Primero, los corregimientos: los nuevos son los que no tienen población en 2010.

Con el conjunto de afectados se calcula el crecimiento del bloque de once corregimientos que el LEEME recomienda comparar como uno solo.

In [14]:
todos = {fila["id_corregimiento"] for fila in poblacion}
nuevos = {fila["id_corregimiento"] for fila in poblacion if fila["pob_2010"] is None}
padres = set(PADRES_LEY_296)
sin_cambios = todos - nuevos - padres
afectados = padres | nuevos

print("Nuevos:", sorted(nuevos))
print("Sin cambios de límites:", sorted(sin_cambios))
print("¿Los nuevos y los padres son disjuntos?", nuevos.isdisjoint(padres))
print("¿Los afectados son subconjunto de todos?", afectados <= todos)

verificar("los nuevos coinciden con los hijos que nombra la Ley 296",
          nuevos, {hijo for hijos in PADRES_LEY_296.values() for hijo in hijos})
verificar("cinco corregimientos no cambiaron de límites", len(sin_cambios), 5)

bloque_2010 = sum(poblacion_por_id[clave]["pob_2010"] or 0 for clave in afectados)
bloque_2023 = sum(poblacion_por_id[clave]["pob_2023"] for clave in afectados)
crecimiento_bloque = round((bloque_2023 / bloque_2010 - 1) * 100, 1)

crecimiento_distrito = round((total_2023 / sum(f["pob_2010"] for f in poblacion if f["pob_2010"]) - 1) * 100, 1)
print(f"Bloque de los once: {bloque_2010:,} -> {bloque_2023:,} ({crecimiento_bloque} %)")
print(f"Distrito completo: {crecimiento_distrito} %")

verificar("el bloque de once creció 11.8 %", crecimiento_bloque, 11.8)
verificar("el distrito creció 21.7 % entre 2010 y 2023", crecimiento_distrito, 21.7)

Nuevos: ['boca_de_tucue', 'candelario_ovalle', 'general_victoriano_lorenzo', 'las_minas', 'riecito', 'san_miguel']
Sin cambios de límites: ['canaveral', 'cocle', 'el_coco', 'penonome_cabecera', 'rio_grande']
¿Los nuevos y los padres son disjuntos? True
¿Los afectados son subconjunto de todos? True
Correcto: los nuevos coinciden con los hijos que nombra la Ley 296
Correcto: cinco corregimientos no cambiaron de límites
Bloque de los once: 43,650 -> 48,786 (11.8 %)
Distrito completo: 21.7 %
Correcto: el bloque de once creció 11.8 %
Correcto: el distrito creció 21.7 % entre 2010 y 2023


Segundo uso: ¿qué indicadores de la base larga (tabla 99) existen en cada año? El nombre del indicador antes de ` | ` es el indicador base. La intersección dice qué se puede seguir en el tiempo; la diferencia, qué aparece solo en un año.

In [15]:
from penonome.carga import cargar_base_larga

base = cargar_base_larga()

def indicadores_de(fecha):
    """Conjunto de indicadores base (sin desagregación) con dato en esa fecha."""
    return {fila["indicador"].split(" | ")[0] for fila in base if fila["fecha"] == fecha}

en_2000, en_2010, en_2023 = indicadores_de("2000"), indicadores_de("2010"), indicadores_de("2023")
comunes = en_2000 & en_2010 & en_2023
solo_2023 = en_2023 - en_2010
del_censo = {i for i in solo_2023 if i.startswith("censo_")}
del_pnud = {i for i in solo_2023 if i.startswith("pnud_")}

print("En los tres censos:", sorted(comunes))
print("Solo en 2023:", len(solo_2023), "indicadores | del Censo:", len(del_censo), "| del PNUD:", len(del_pnud))
print("Los demás:", sorted(solo_2023 - del_censo - del_pnud))
print("Indicadores de 2010 que no están en 2023:", en_2010 - en_2023)

verificar("entre censos solo la población es comparable", comunes, {"poblacion"})
verificar("las tres familias de 2023 son el Censo, el IPM y la superficie",
          del_censo | del_pnud | {"superficie"}, solo_2023)
verificar("los indicadores de 2010 son un subconjunto de los de 2023", en_2010 <= en_2023, True)

En los tres censos: ['poblacion']
Solo en 2023: 24 indicadores | del Censo: 20 | del PNUD: 3
Los demás: ['superficie']
Indicadores de 2010 que no están en 2023: set()
Correcto: entre censos solo la población es comparable
Correcto: las tres familias de 2023 son el Censo, el IPM y la superficie
Correcto: los indicadores de 2010 son un subconjunto de los de 2023


Tercer uso: cruzar dos condiciones de los corregimientos. Una es la **pobreza multidimensional alta** (incidencia H de al menos 50 %, según el PNUD, 2023); la otra, tener **registros de daños** del sismo en la compilación de prensa de mapadelterremoto.com. Cada condición es un conjunto de claves, y los operadores responden preguntas como «¿cuáles cumplen las dos?».

**Error común:** `registros_danos` cuenta registros de prensa, no viviendas dañadas, y que un corregimiento no tenga registros no prueba que no haya sufrido daños: puede ser que la prensa no llegara.

In [16]:
ipm = cargar_ipm()
registros = cargar_registros_danos()

con_registros = {f["id_corregimiento"] for f in registros if f["registros_danos"] > 0}
pobreza_alta = {f["id_corregimiento"] for f in ipm if f["incidencia_h"] >= 50}

print("Con registros de daños:", sorted(con_registros))
print("Pobreza multidimensional alta:", sorted(pobreza_alta))
print("Ambas condiciones (&):", sorted(con_registros & pobreza_alta))
print("Pobreza alta y sin registros (-):", sorted(pobreza_alta - con_registros))
print("Con registros y pobreza menor (-):", len(con_registros - pobreza_alta), "corregimientos")
print("Cumplen una u otra (|):", len(con_registros | pobreza_alta), "| solo una de las dos (^):", len(con_registros ^ pobreza_alta))

verificar("tres corregimientos tienen pobreza alta y registros de daños",
          con_registros & pobreza_alta, {"chiguiri_arriba", "san_miguel", "tulu"})
verificar("hay siete corregimientos con pobreza alta", len(pobreza_alta), 7)
verificar("la unión es la suma menos la intersección",
          len(con_registros | pobreza_alta), len(con_registros) + len(pobreza_alta) - len(con_registros & pobreza_alta))

Con registros de daños: ['chiguiri_arriba', 'cocle', 'el_coco', 'las_minas', 'pajonal', 'penonome_cabecera', 'rio_grande', 'san_miguel', 'toabre', 'tulu']
Pobreza multidimensional alta: ['boca_de_tucue', 'candelario_ovalle', 'chiguiri_arriba', 'riecito', 'rio_indio', 'san_miguel', 'tulu']
Ambas condiciones (&): ['chiguiri_arriba', 'san_miguel', 'tulu']
Pobreza alta y sin registros (-): ['boca_de_tucue', 'candelario_ovalle', 'riecito', 'rio_indio']
Con registros y pobreza menor (-): 7 corregimientos
Cumplen una u otra (|): 14 | solo una de las dos (^): 11
Correcto: tres corregimientos tienen pobreza alta y registros de daños
Correcto: hay siete corregimientos con pobreza alta
Correcto: la unión es la suma menos la intersección


`zip` une dos secuencias elemento a elemento. Se usa para emparejar el porcentaje de viviendas sin internet con la incidencia de pobreza multidimensional de cada corregimiento y ver si van juntos.

**Ampliación (no está en el PDF):** el IPM resume la pobreza en tres números: la incidencia H (porcentaje de personas pobres), la intensidad A (porcentaje de carencias que sufren esas personas) y el índice M0, que es H x A. Como el PNUD publica H y A redondeados, M0 coincide con su producto solo con una diferencia pequeña.

In [17]:
incidencia = {fila["id_corregimiento"]: fila["incidencia_h"] for fila in ipm}
claves = [fila["id_corregimiento"] for fila in viviendas]
pares = dict(zip(claves, zip([sin_internet[c] for c in claves], [incidencia[c] for c in claves])))

for clave in ("penonome_cabecera", "pajonal", "tulu", "riecito"):
    print(f"{clave:<20} sin internet {pares[clave][0]:>5} %   pobreza multidimensional {pares[clave][1]:>3} %")

minima = min(incidencia[clave] for clave in mayoria_sin_internet)
print("Incidencia mínima entre los corregimientos con más del 90 % sin internet:", minima, "%")

verificar("Tulú: 99.4 % sin internet y 76 % de pobreza multidimensional", pares["tulu"], (99.4, 76))
verificar("M0 coincide con H x A salvo el redondeo",
          all(abs(f["ipm_m0"] - f["incidencia_h"] * f["intensidad_a"] / 10000) <= 0.005 for f in ipm), True)

penonome_cabecera    sin internet  17.5 %   pobreza multidimensional  16 %
pajonal              sin internet  57.6 %   pobreza multidimensional  45 %
tulu                 sin internet  99.4 %   pobreza multidimensional  76 %
riecito              sin internet  98.2 %   pobreza multidimensional  86 %
Incidencia mínima entre los corregimientos con más del 90 % sin internet: 71 %
Correcto: Tulú: 99.4 % sin internet y 76 % de pobreza multidimensional
Correcto: M0 coincide con H x A salvo el redondeo


## Conclusiones

- Penonomé tenía **104,326 habitantes** en 2023, un **21.7 %** más que en 2010. La cabecera, El Coco y Cañaveral concentran el 44.3 %.
- Entre los corregimientos que no cambiaron de límites, **El Coco** casi duplicó su población (**+97.4 %**). Los cinco que se dividieron por la Ley 296 de 2022 parecen perder población, pero al sumar sus corregimientos nuevos el bloque creció **11.8 %**: la caída aparente es un efecto de los límites, no de la migración.
- El **41.9 %** de las viviendas del distrito no tiene internet (44.0 % en Coclé). La brecha es territorial: 17.5 % en la cabecera frente a más del 90 % en cinco corregimientos rurales, con Tulú en 99.4 %. Esos cinco también tienen una incidencia de pobreza multidimensional de **71 %** o más, frente a 16 % en la cabecera.
- SINAPROC evaluó **2,531 viviendas** en Coclé tras el sismo: 414 con daño total y **1,750 (69.1 %)** con daño parcial. De las 431 personas con discapacidad registradas, 314 viven en viviendas con daño parcial y 77 en viviendas con daño total; de las 21 mujeres embarazadas, 11 y 7. El diccionario doble por nivel de daño suma exactamente los totales publicados.
- Tres corregimientos (Chiguirí Arriba, San Miguel y Tulú) combinan pobreza multidimensional alta con registros de daños en la prensa. Otros cuatro con pobreza alta (Boca de Tucué, Candelario Ovalle, Riecito y Río Indio) no tienen registros, lo que no prueba que no hayan sufrido daños.
- Entre censos, el único indicador comparable del paquete es la población. Los indicadores de vivienda y pobreza son solo de 2023: describen la vulnerabilidad previa al sismo, no su cambio posterior.

[Proyecto](../../README.md) | [Anterior](../../README.md) | [Siguiente](s02-funciones-clausuras-clases.ipynb)